<a href="https://colab.research.google.com/github/rylam11/BUS4-118-Prompt-Engineering/blob/main/Ryan_Lam_Exercise2_ReACT_Code_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2: Code Generation with ReACT Prompting

## Goal
Use a ReACT-style process to generate Python code, run the code, observe the results, and improve the code if needed.

## Tools Used
- Google Colab
- Python
- Gemini API

## Coding Task
Create a Python program that calculates the average of a list of customer order amounts and handles invalid or empty input.

## ReACT Process
1. Reason/Plan what the code should do
2. Generate the Python code
3. Run the code
4. Observe the output
5. Fix or improve the code if needed

In [ ]:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata
import time

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def ask_gemini(prompt):
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt
            )
            return response.text
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < 2:
                time.sleep(5)

    return "Unable to get a response after 3 attempts."

print("Gemini setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
Gemini setup complete.


## ReACT Prompt


In [ ]:
react_prompt = """
You are a Python coding assistant using a ReACT-style process.

Task:
Create a Python program that calculates the average of a list of customer order amounts.

Requirements:
- Use a Python function.
- Accept a list of values as input.
- Ignore values that are not numbers.
- Handle an empty list without crashing.
- Return the average rounded to 2 decimal places.
- If there are no valid numbers, return a clear message instead of causing an error.

Follow this process:

1. Reason/Plan:
Briefly explain what the code should do and what edge cases must be handled.

2. Generate Code:
Write the Python code.

3. Test Plan:
Provide at least 3 test cases, including:
- a normal list of numbers
- a list containing an invalid value
- an empty list

Do not skip any section.
"""

react_output = ask_gemini(react_prompt)

print(react_output)

### 1. Reason/Plan

**Objective:** Create a Python function to calculate the average of a list of customer order amounts.

**Requirements & Edge Cases to Handle:**
- **Function Input:** A list containing mixed data types (numbers, strings, booleans, etc.).
- **Filtering:** Use `isinstance(val, (int, float))` and ensure we exclude booleans (since in Python, `isinstance(True, int)` evaluates to `True`) to strictly filter for valid numbers.
- **Empty/Invalid Lists:** If the list is empty or contains no valid numbers after filtering, the function should return a clear, user-friendly message rather than crashing (e.g., preventing a `ZeroDivisionError`).
- **Rounding:** The final average must be rounded to 2 decimal places using `round(value, 2)`.

---

### 2. Generate Code

```python
def calculate_average_order(orders):
    """
    Calculates the average of a list of customer order amounts.
    Ignores non-numeric values and handles empty or invalid lists gracefully.
    """
    if not orde

## Execute and Observe

I copied the generated Python code into Colab and ran the test cases to check whether the program worked as expected.

In [ ]:
def calculate_average_order(orders):
    """
    Calculates the average of a list of customer order amounts.
    Ignores non-numeric values and handles empty or invalid lists gracefully.
    """
    if not orders:
        return "No orders provided."

    valid_orders = []

    for item in orders:
        if isinstance(item, (int, float)) and not isinstance(item, bool):
            valid_orders.append(item)

    if not valid_orders:
        return "No valid numeric orders found."

    average = sum(valid_orders) / len(valid_orders)

    return round(average, 2)


# Test Case 1: Normal list
normal_orders = [45.50, 100.00, 25.25]
print("Test 1:", calculate_average_order(normal_orders))

# Test Case 2: Invalid values included
mixed_orders = [50.0, "invalid", 150.0, None, True, 25.0]
print("Test 2:", calculate_average_order(mixed_orders))

# Test Case 3: Empty list
empty_orders = []
print("Test 3:", calculate_average_order(empty_orders))

Test 1: 56.92
Test 2: 75.0
Test 3: No orders provided.


## Observation

The generated code passed all three test cases. However, I noticed that it would still accept negative numbers as valid customer order amounts. Since an order amount should not normally be negative, I decided to improve the code by ignoring negative values.

In [ ]:
improvement_prompt = f"""
You are a Python coding assistant using a ReACT-style process.

The following code successfully passed its original tests:

{'''
def calculate_average_order(orders):
    if not orders:
        return "No orders provided."

    valid_orders = []

    for item in orders:
        if isinstance(item, (int, float)) and not isinstance(item, bool):
            valid_orders.append(item)

    if not valid_orders:
        return "No valid numeric orders found."

    average = sum(valid_orders) / len(valid_orders)
    return round(average, 2)
'''}

Observation:
The code accepts negative numbers as valid order amounts.

Improve the code so that:
- negative numbers are ignored
- non-numeric values are still ignored
- empty input is handled
- the average is rounded to 2 decimal places

Respond with:
1. Brief Fix Explanation
2. Revised Python Code
"""

improved_output = ask_gemini(improvement_prompt)

print(improved_output)

1. **Brief Fix Explanation**:
To ignore negative numbers in addition to non-numeric values and booleans, we simply update the condition inside the `for` loop to check that the item is greater than or equal to `0`. All other requirements (handling empty input, filtering out non-numeric/boolean values, and rounding to 2 decimal places) remain unchanged.

2. **Revised Python Code**:
```python
def calculate_average_order(orders):
    if not orders:
        return "No orders provided."

    valid_orders = []

    for item in orders:
        if isinstance(item, (int, float)) and not isinstance(item, bool) and item >= 0:
            valid_orders.append(item)

    if not valid_orders:
        return "No valid numeric orders found."

    average = sum(valid_orders) / len(valid_orders)
    return round(average, 2)
```


## Test the Improved Code

I ran the revised code again and added a test containing a negative order amount to confirm that the improvement worked.

In [ ]:
def calculate_average_order(orders):
    if not orders:
        return "No orders provided."

    valid_orders = []

    for item in orders:
        if isinstance(item, (int, float)) and not isinstance(item, bool) and item >= 0:
            valid_orders.append(item)

    if not valid_orders:
        return "No valid numeric orders found."

    average = sum(valid_orders) / len(valid_orders)
    return round(average, 2)


# Test improved code
negative_test = [50, -25, 100]

print("Improved Test:", calculate_average_order(negative_test))

Improved Test: 75.0


## Exercise 2 Summary

The ReACT process was used to plan, generate, run, observe, and improve Python code.

The first version successfully calculated the average, ignored non-numeric values, and handled empty input. After observing the results, I identified an additional edge case: negative numbers were still being treated as valid order amounts.

I revised the code so negative values are ignored. The improved test returned 75.0 for the list [50, -25, 100], confirming that only the valid positive order amounts were included.